In [0]:
"""
Fraud Detection - Using Your Sample Transactions
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from datetime import datetime
from typing import Dict, Tuple

# ============================================
# 1. LOAD YOUR EXISTING DATA
# ============================================

print("="*60)
print("FRAUD DETECTION: ML-Based System")
print("="*60)

# Read your existing transactions file
tx_path = "/Workspace/Users/molugurikoushik68@gmail.com/banking-agent/data/sample_transactions.csv"

print(f"\nLoading transactions from: {tx_path}")

try:
    df = pd.read_csv(tx_path)
    print(f"✓ Loaded {len(df)} transactions")
    print(f"\nColumns: {list(df.columns)}")
    print(f"Fraud distribution: {df['is_fraudulent'].value_counts().to_dict()}")
except Exception as e:
    print(f"✗ Error loading file: {e}")
    print(f"Creating sample data instead...")
    
    # Fallback: create sample data
    np.random.seed(42)
    df = pd.DataFrame({
        'transaction_id': range(1, 101),
        'amount': np.random.uniform(10, 1000, 100),
        'timestamp': pd.date_range('2024-01-01', periods=100, freq='H'),
        'merchant': np.random.choice(['Amazon', 'Walmart', 'Target', 'Unknown'], 100),
        'is_fraudulent': np.random.choice([0, 0, 0, 0, 1], 100)
    })

# ============================================
# 2. ENGINEER FEATURES FROM RAW DATA
# ============================================

print("\n" + "="*60)
print("ENGINEERING FEATURES FROM RAW DATA")
print("="*60)

def engineer_features(df):
    """
    Convert raw transaction data into ML features.
    
    Raw data:
    - amount (10-1000)
    - timestamp (dates/times)
    - merchant (text names)
    - is_fraudulent (label)
    
    Engineered features:
    - amount (numeric)
    - is_new_merchant (was this merchant seen before?)
    - is_unusual_time (is it after hours?)
    - transaction_frequency (how many transactions from this merchant?)
    - days_since_last (how long since last transaction?)
    - is_international (for now, we'll infer from unknown merchants)
    """
    
    engineered = df.copy()
    
    # 1. Amount feature (already numeric)
    engineered['amount'] = df['amount'].astype(float)
    
    # 2. New merchant feature (is this an unknown merchant?)
    engineered['is_new_merchant'] = (df['merchant'] == 'Unknown').astype(int)
    
    # 3. Unusual time feature (is timestamp in off-hours? 10 PM - 6 AM)
    engineered['timestamp'] = pd.to_datetime(df['timestamp'])
    engineered['hour'] = engineered['timestamp'].dt.hour
    engineered['is_unusual_time'] = ((engineered['hour'] >= 22) | (engineered['hour'] < 6)).astype(int)
    
    # 4. Transaction frequency (how often does this merchant appear?)
    merchant_counts = df['merchant'].value_counts().to_dict()
    engineered['frequency'] = df['merchant'].map(merchant_counts).fillna(1)
    
    # 5. Days since last transaction (time gap)
    engineered['days_since_last_transaction'] = engineered['timestamp'].diff().dt.days.fillna(0)
    
    # 6. International (assume Unknown = potentially international)
    engineered['is_international'] = (df['merchant'] == 'Unknown').astype(int)
    
    return engineered

engineered_df = engineer_features(df)

print(f"\n✓ Features engineered!")
print(f"\nSample engineered data:")
print(engineered_df[['amount', 'is_new_merchant', 'is_unusual_time', 'frequency']].head())

# ============================================
# 3. TRAIN/TEST SPLIT (Proper ML Practice)
# ============================================

print("\n" + "="*60)
print("TRAINING FRAUD DETECTION MODEL")
print("="*60)

from sklearn.model_selection import train_test_split

# Select features
feature_columns = ['amount', 'is_new_merchant', 'is_unusual_time', 'frequency', 'days_since_last_transaction', 'is_international']
X = engineered_df[feature_columns]
y = engineered_df['is_fraudulent']

print(f"\nTraining features: {feature_columns}")

# IMPORTANT: Split data into training and testing sets
# Training set (80%): Used to train the model
# Testing set (20%): Used to test on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,        # 20% for testing (unseen data)
    random_state=42,
    stratify=y            # Keep fraud ratio same in train/test
)

print(f"\nData split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Testing samples: {len(X_test)}")
print(f"  Training fraud rate: {y_train.mean():.1%}")
print(f"  Testing fraud rate: {y_test.mean():.1%}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # IMPORTANT: Use same scaler on test data

# Train model (on training data only!)
model = RandomForestClassifier(
    n_estimators=50,
    max_depth=8,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# Evaluate on BOTH training and testing data
train_accuracy = model.score(X_train_scaled, y_train)
test_accuracy = model.score(X_test_scaled, y_test)

print(f"\n✓ Model trained!")
print(f"\nAccuracy Comparison:")
print(f"  Training accuracy (80 samples): {train_accuracy:.2%}")
print(f"  Testing accuracy (20 samples):  {test_accuracy:.2%}")
print(f"  Difference: {(train_accuracy - test_accuracy):.2%}")

if train_accuracy - test_accuracy > 0.1:
    print(f"  ⚠️ Note: Large gap suggests some overfitting")
else:
    print(f"  ✓ Good generalization (low overfitting)")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature Importance:")
for idx, row in feature_importance.iterrows():
    print(f"  {row['feature']}: {row['importance']:.2%}")

# ============================================
# 4. PREDICT FRAUD
# ============================================

def predict_fraud(transaction: Dict, model, scaler, feature_columns) -> Tuple[float, Dict]:
    """Score a transaction for fraud"""
    
    features = [transaction.get(f, 0) for f in feature_columns]
    features_scaled = scaler.transform([features])
    fraud_probability = model.predict_proba(features_scaled)[0][1]
    
    # Explanation
    risk_factors = []
    if transaction.get('amount', 0) > 500:
        risk_factors.append("Large amount")
    if transaction.get('is_new_merchant') == 1:
        risk_factors.append("New merchant")
    if transaction.get('is_unusual_time') == 1:
        risk_factors.append("Off-hours")
    
    if fraud_probability > 0.75:
        action = "BLOCK"
        risk = "VERY HIGH"
    elif fraud_probability > 0.5:
        action = "VERIFY"
        risk = "HIGH"
    else:
        action = "ALLOW"
        risk = "LOW"
    
    return fraud_probability, {
        'risk_level': risk,
        'action': action,
        'factors': risk_factors
    }

# ============================================
# 5. TEST FRAUD DETECTION
# ============================================

print("\n" + "="*60)
print("TESTING FRAUD DETECTION")
print("="*60)

test_cases = [
    {
        'name': 'Normal - Small amount, known merchant',
        'amount': 50,
        'is_new_merchant': 0,
        'is_unusual_time': 0,
        'frequency': 10,
        'days_since_last_transaction': 1,
        'is_international': 0
    },
    {
        'name': 'Suspicious - Large amount, new merchant, off-hours',
        'amount': 800,
        'is_new_merchant': 1,
        'is_unusual_time': 1,
        'frequency': 1,
        'days_since_last_transaction': 30,
        'is_international': 1
    }
]

for test in test_cases:
    name = test.pop('name')
    score, explanation = predict_fraud(test, model, scaler, feature_columns)
    
    print(f"\n{name}")
    print(f"  Fraud Score: {score:.0%}")
    print(f"  Risk Level: {explanation['risk_level']}")
    print(f"  Action: {explanation['action']}")
    print(f"  Factors: {', '.join(explanation['factors']) if explanation['factors'] else 'None'}")

print("\n" + "="*60)
print("✓ FRAUD DETECTION READY")
print("="*60)

# Export
fraud_model = model
fraud_scaler = scaler
fraud_features = feature_columns

print(f"\n✓ Exported for Phase 3:")
print(f"  - fraud_model")
print(f"  - fraud_scaler")
print(f"  - fraud_features")